# 실전 — 실제 데이터로 검증: 실제 HTTP 공격 페이로드로 검증하기 (ch15)

ch15 의 char-ngram TF-IDF 파이프라인을 **실제 공개 데이터셋**(HTTP params, SQLi·XSS·cmdi·path-traversal 라벨) 에 그대로 얹어, 합성에서 배운 것이 실 페이로드에서도 성립하는지 검증한다.

> **인터넷 필요**: 실행 시 데이터(약 2MB)를 내려받는다.

핵심 관찰 세 가지를 미리 짚는다. 첫째, char-ngram(2~4) TF-IDF 는 실제 페이로드에서도 공격/정상을 강하게 분리한다. 둘째, 그러나 **희소 공격 유형(cmdi ~90건)의 recall 은 낮아** ch15·ch16 의 클래스 불균형 교훈이 실데이터에서 그대로 재현된다. 셋째, **인코딩 회피**는 정규화 없이는 탐지를 무너뜨린다 — ch15 §15.2 의 정규화가 왜 필수인지 실측으로 확인한다.

## 데이터셋 이야기 — 어떻게 수집됐나 (HTTP 공격 페이로드)

스페인 CSIC 의 HTTP 데이터셋 계열에서 유래한 **실제 웹 요청 파라미터** 모음이다. 자동 생성기와 실제 트래픽으로 만든 정상 요청과, SQL 인젝션·XSS·명령 주입·경로 탐색 공격 페이로드를 **유형 라벨과 함께** 담는다.

**시나리오**: WAF 가 받는 원시 요청 문자열만으로 정상과 공격을, 나아가 공격 유형까지 구분할 수 있는가.

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# !pip install pandas scikit-learn matplotlib

from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

import urllib.parse, urllib.request, ssl, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, recall_score

## 1. 실데이터 내려받기

실제 HTTP 요청 파라미터에 정상/공격 라벨과 공격 유형(sqli·xss·cmdi·path-traversal) 이 붙은 공개 데이터셋을 받는다.

In [ ]:
URL = 'https://raw.githubusercontent.com/Morzeux/HttpParamsDataset/master/payload_full.csv'
def fetch(fname, url):
    if not os.path.exists(fname):
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=60, context=ssl.create_default_context()) as r:
            open(fname, 'wb').write(r.read())
    return fname

df = pd.read_csv(fetch('http_params.csv', URL)).dropna(subset=['payload'])
print('shape:', df.shape)
print(df['label'].value_counts())
print(df['attack_type'].value_counts())

## 2. 정규화 (ch15 §15.2)

URL 이중 디코딩 + 소문자화. 인코딩으로 숨긴 공격 문자를 드러내는 전처리다.

In [ ]:
def normalize(s):
    s = str(s)
    for _ in range(2):
        s = urllib.parse.unquote_plus(s)
    return s.lower()

df['norm'] = df['payload'].map(normalize)
print('예:', repr(df['payload'].iloc[df[df.attack_type=="sqli"].index[0]])[:60])
print('정규화 후:', repr(df['norm'].iloc[df[df.attack_type=="sqli"].index[0]])[:60])

## 3. char-ngram TF-IDF + 학습 (ch15 §15.3.3)

단어 토큰이 아니라 **문자 2~4-gram** 으로 벡터화한다 — `union` 이라는 단어보다 `un`·`ni`·`io` 같은 부분 문자열 빈도가 공격 시그니처다.

In [ ]:
# ── 실습 과제 ──
# 목표: 3. char-ngram TF-IDF + 학습 (ch15 §15.3.3) — 단어 토큰이 아니라 **문자 2~4-gram** 으로 벡터화한다 — `union` 이라는 단어보다 `un`·`ni`·`io` 같은 부분 문자열 빈도가 공격 시그니처다.
# 만들 결과: X, y_bin, y_type, vec, Xtr_v, Xte_v
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
X = ...
y_bin = ...
y_type = ...
Xtr, Xte, ytr, yte, atr, ate = ...
    X, y_bin, y_type, test_size = ...

vec = ...
Xtr_v = ...
Xte_v = ...
...

clf = ...
proba = ...
pred = ...
print(classification_report(yte, pred, target_names = ...
...
```
</details>


## 4. 공격 유형 분류 — 희소 클래스의 벽

이진 탐지는 거의 완벽하지만, **희소 공격 유형(cmdi 등)의 recall 은 낮다.** ch15·ch16 의 클래스 불균형 교훈이 실데이터에서 그대로 나타난다.

In [ ]:
# ── 실습 과제 ──
# 목표: 4. 공격 유형 분류 — 희소 클래스의 벽 — 이진 탐지는 거의 완벽하지만, **희소 공격 유형(cmdi 등)의 recall 은 낮다.** ch15·ch16 의 클래스 불균형 교훈이 실데이터에서 그대로 나타난다.
# 만들 결과: rf, class_weight, pred_type
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
rf = ...
                            class_weight = ...
pred_type = ...
print(classification_report(ate, pred_type, digits = ...
```
</details>


## 5. 인코딩 회피 — 정규화가 왜 필수인가 (ch15 §15.8)

SQLi 페이로드를 **URL 인코딩**해 숨겼을 때, 정규화 없이 raw 로 넣으면 재현율이 급락하고, 정규화를 거치면 다시 잡힌다.

In [ ]:
sqli_test = Xte[ate == 'sqli']            # 이미 정규화된 SQLi 테스트 payload
raw_sqli = df.loc[sqli_test.index, 'payload']  # 원본(인코딩 가능)
# 공격을 한 번 더 URL 인코딩해 숨김
evaded = raw_sqli.map(lambda s: urllib.parse.quote(str(s)))

# (A) 정규화 없이 raw evaded 그대로
r_raw = recall_score(np.ones(len(evaded)), clf.predict(vec.transform(evaded)))
# (B) ch15 정규화 후
r_norm = recall_score(np.ones(len(evaded)), clf.predict(vec.transform(evaded.map(normalize))))
print(f'인코딩 회피 SQLi 재현율 — 정규화 없음: {r_raw:.3f}  /  정규화 후: {r_norm:.3f}')

## 6. 정리 — 실 페이로드에서 확인한 것

1. **char-ngram TF-IDF 는 실 페이로드에서도 강력하다** — 정상/공격 이진 분리는 거의 완벽하다.
2. **희소 공격 유형은 여전히 어렵다** — cmdi 처럼 표본이 적은 유형은 recall 이 낮아, 클래스별 지표(ch16)와 데이터 보강이 필요하다.
3. **정규화 없는 탐지는 인코딩 회피에 무너진다** — ch15 §15.2 의 이중 디코딩이 왜 파이프라인의 첫 단계인지 실측으로 드러난다.

> 같은 코드로 HTTP CSIC 2010 전체나 다른 웹 공격 데이터셋으로 확장해 볼 수 있다. 벡터화·모델 골격은 동일하다.